# Is the information in the frozen encoder vectors?

**Question.** The frozen encoder outputs (MiniLM for text, UniChart's encoder for the image) are lists of numbers nobody can read. Do they contain what the task needs?

**Method.** A *probe* is a very small classifier trained on the frozen vectors to predict one property. If it cannot beat the **baseline** (always guessing the commonest answer), the property is not easily readable from the vector. Every probe also has a **shuffled-label control** (trained on scrambled answers, it should fall to the baseline). Nothing here changes the main pipeline: `probes/run_probes.py` only reads the saved vectors and tables.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name in ("probes", "data_exploration") else Path.cwd()
results = json.load(open(ROOT / "probes/results/probe_results.json"))

r = results["retrieval"]
print(f"Retrieval with no trained part at all ({r['questions']} extract questions, training subset)")
print(f"  closest cell to the question is the answer cell: {r['top1'] * 100:.1f}%  (chance {r['chance_top1'] * 100:.1f}%)")
print(f"  answer cell among the 3 closest: {r['top3'] * 100:.1f}%, mean reciprocal rank {r['mean_reciprocal_rank']:.2f}")
by = pd.DataFrame(r["by_answer_cell"]).T
by["top1"] = (by["top1"] * 100).round(1)
by["chance_top1"] = (by["chance_top1"] * 100).round(1)
by.columns = ["questions", "closest cell is right (%)", "chance (%)"]
display(by.rename_axis("the answer is a ... cell"))

Retrieval with no trained part at all (2014 extract questions, training subset)
  closest cell to the question is the answer cell: 41.4%  (chance 5.9%)
  answer cell among the 3 closest: 57.3%, mean reciprocal rank 0.53


,questions,closest cell is right (%),chance (%)
the answer is a ... cell,,,
value,1362.0,57.6,6.0
row_label,550.0,5.6,5.7
header,102.0,17.6,4.6


**Reading the retrieval result.** When the answer is a **value cell**, plain similarity finds it 58% of the time, because the cell text repeats words from the question. When the answer is a **row name** (for example "which country has the highest rate?") it is **at chance (5.6% against 5.7%)**: the answer cell shares no words with the question, and comparing values is something the vectors do not do. Header cells sit in between.

In [2]:
rows = []
for name, v in results.items():
    if name == "retrieval":
        continue
    rows.append({"probe": name, "baseline": v["baseline"] * 100, "linear probe": v["linear"] * 100, "small MLP probe": v["mlp"] * 100,
                 "shuffled-label control": v["control_shuffled"] * 100, "test items": v["test"]})
table = pd.DataFrame(rows).set_index("probe").round(1)
table["gain over baseline (MLP)"] = (table["small MLP probe"] - table["baseline"]).round(1)
display(table)

,baseline,linear probe,small MLP probe,shuffled-label control,test items,gain over baseline (MLP)
probe,,,,,,
number order (which cell is larger),48.8,60.3,63.2,48.7,3325,14.4
number size (power of ten of a value),44.7,72.5,74.6,44.3,5908,29.9
question intent (control),83.3,90.9,92.5,76.2,252,9.2
calculation type (from the question),30.0,86.7,86.7,36.7,30,56.7
"image, whole-chart vector: data series: 1, 2 or 3+",73.1,94.2,94.4,62.0,553,21.3
"image, 4x4 grid vectors: data series: 1, 2 or 3+",73.1,92.4,94.2,47.9,553,21.1
"image, whole-chart vector: rows: up to 5, 6 to 10, 11+",36.5,89.7,91.1,30.9,553,54.6
"image, 4x4 grid vectors: rows: up to 5, 6 to 10, 11+",36.5,89.5,91.9,31.8,553,55.4
"image, whole-chart vector: values shown as percentages or not",59.7,98.2,98.2,52.3,553,38.5


## What the probes show

**Frozen text vectors (MiniLM)**
- **Numbers: size yes, order barely.** A probe reads the *size* of a number (power of ten) at 73 to 75% against a 45% baseline, so the vector knows roughly how big a value is. But asked which of two cells in a column is **larger**, it reaches only **60 to 63% against 50% chance**. Fine-grained numeric order is not readable. This matches what we saw in training: the model needed hand-made rank features, and adding match features lifted accuracy from 38% to 48%.
- **Questions: yes.** The question vector predicts the kind of answer (91 to 93% against an 83% baseline) and the type of calculation (87% against 30%, only 30 test questions). This is the **control that works**: the probes are not broken, and the router can do its job.

**Frozen image vectors (UniChart's encoder)**
- The chart image vector predicts the **number of data series** (94% vs 73%), the **number of rows** (90 to 91% vs 37%) and **whether values are percentages** (98% vs 60%). The 4x4 grid vectors do no better than the whole-chart vector.
- **Caution:** these targets were read from UniChart's own table for the chart, and the same encoder produced that table. So the structure *has* to be in the vector. This shows the image vector is rich in chart structure. It does **not** show that image vectors are insufficient.

## What this does and does not support

- **Supported:** frozen text vectors cannot compare numbers or rank cells, so superlative and comparison questions cannot be solved from raw vectors alone and need extra structure (explicit numeric features, or a model that reads the whole table).
- **Not supported:** that the image vectors lack chart structure. They have it. If the image did not help our model, the cause is more likely how it is fused, or things not probed here.
- **Limits:** a weak probe means "not easily readable", not "absent". Test sets are small for some probes (30 for calculation type). Targets for the image probes come from extracted tables, so they are approximate.

## TO DO

- Probes for **colour and series linkage**, **chart type** and **position**: ChartQA has no labels for these, so about 100 charts need labelling by hand (or a rough stand-in).
- Compare with an **oracle**: give the heads the true chart metadata (type, legend, series colours) and see how much accuracy rises on the questions that need it. A big rise would show the raw vectors lack that information.
- Probe whether the information is **used**, not just present: does the trained model's accuracy drop when the image features are removed from the questions that depend on structure?

## Which parts of the input are lost? (input-level probes)

The probes above were about information in general. These ask about **specific parts of the input**: the extracted table (read one cell at a time by MiniLM) and the chart image (UniChart's encoder, kept as one whole-chart vector, a 4x4 grid or an 8x8 grid). The answer key is read from the extracted table. Script: `probes/run_input_probes.py`.

In [3]:
inp = json.load(open(ROOT / "probes/results/input_probe_results.json"))
rows = []
for name, v in inp.items():
    rows.append({"probe": name, "baseline": v["baseline"] * 100, "linear": v["linear"] * 100,
                 "MLP": (v["mlp"] * 100) if v["mlp"] is not None else None, "shuffled control": v["control_shuffled"] * 100, "test items": v["test"]})
tbl = pd.DataFrame(rows).set_index("probe").round(1)
tbl["gain over baseline (best probe)"] = (tbl[["linear", "MLP"]].max(axis=1) - tbl["baseline"]).round(1)
pd.set_option("display.max_colwidth", 90)
display(tbl)

,baseline,linear,MLP,shuffled control,test items,gain over baseline (best probe)
probe,,,,,,
text: leading digit of the number (1 to 9),24.9,62.8,69.8,22.9,5994,44.9
"text: number of decimal places (0, 1, 2+)",47.5,83.4,85.9,43.8,5994,38.4
"text: position of a row name (first, middle, last third)",36.9,51.2,52.8,34.7,5079,15.9
text: is this cell the largest in its column (50/50),49.5,58.6,57.8,48.3,1395,9.1
"image, whole-chart vector: size of the largest value (power of ten)",51.5,93.6,94.0,33.9,499,42.5
"image, 4x4 grid: size of the largest value (power of ten)",51.5,91.6,93.4,18.2,499,41.9
"image, 8x8 grid: size of the largest value (power of ten)",51.5,90.6,NaN,19.6,499,39.1
"image, whole-chart vector: is the last row's value above the first row's (series 1)",75.6,80.8,85.2,66.5,499,9.6
"image, 4x4 grid: is the last row's value above the first row's (series 1)",75.6,79.2,86.8,56.3,499,11.2


### What is kept and what is lost

| Part of the input | Kept in the frozen vector? | Evidence |
|---|---|---|
| **Table cell: the words** (row and column names) | Partly | closest cell is right 58% of the time for value cells (earlier probe) |
| **Table cell: the digits** | **Yes** | leading digit read at 63 to 70% (baseline 25%); number of decimals 83 to 86% (baseline 48%); size of the number 73 to 75% |
| **Table cell: its order against other cells** | **No** | which of two is larger 60 to 63% (chance 50%); is this the largest in its column 58% (chance 50%) |
| **Table cell: its position in the table** | **Mostly no** | first, middle or last third of the rows 51 to 53% (baseline 37%) |
| **Image: overall scale of the data** | **Yes** | size of the largest value 91 to 94% (baseline 52%) |
| **Image: number of series and rows, percentages** | **Yes** | 94%, 90 to 91%, 98% (earlier probes) |
| **Image: shape of the data** (which bar is largest, trend) | **Weakly** | where the largest value is 75 to 83% (baseline 67%); trend up or down 76 to 87% (baseline 76%) |
| **Image: colours, legend, chart type, position of bars, axis labels** | Not tested | needs hand labels |
| **Chart title, axis names, units, legend** | **Not in the input** | the extracted table only has "Characteristic" and values |

### The pattern

In both text and image, the frozen vectors keep **the content of individual items** (the words, the digits, the overall scale) and lose **how items relate to each other and where they sit**: which value is bigger, which cell is largest, where a row is in the table, which bar is highest, which way a trend goes.

### Limits

- **Finer image grids did not help.** The 8x8 grid scored no better than the single whole-chart vector, but its linear probe has 66,000 inputs for about 2,000 training charts, so this is inconclusive: it cannot yet separate "the encoder lacks the detail" from "our pooling threw it away".
- The image answer keys come from UniChart's own table, and the test sets are small (about 500 charts). A strong baseline (trend: 76%) leaves little room.
- "Row position" probes the row name alone; a table read as a whole would know the position.